In [11]:
# Import the basic spark library
from pyspark.sql import SparkSession
import pyspark
import findspark 
findspark.init()
findspark.find()
from pyspark import SparkContext, SparkConf
from pyspark.sql import SparkSession


# Create an entry point to the PySpark Application
spark = SparkSession.builder \
      .master("local") \
      .appName("ProjSMBUD") \
      .getOrCreate()
# master contains the URL of your remote spark instance or 'local'

In [12]:
# Load DataFrames (we prefer directly them to the rdd, because we already had csv with headers)
#Configuring them with some examples to check the correctness (now commented before and after the casting)

df_incollection = spark.read.option("header", True).option("delimiter", ";")\
                .csv("incollectionFormatted.csv.txt")
df_book = spark.read.option("header", True).option("delimiter", ";")\
                .csv("bookFormatted.csv.txt")
df_author = spark.read.option("header", True).option("delimiter", ",")\
                .csv("authorFormatted.csv.txt")

#Load bridge tables? (No, now they are in the entities), #they had already id string types, but now we don't have them





# We use of datatypes from pyspark.sql.types
from pyspark.sql.types import StringType, DateType, FloatType, StructType, StructField, ArrayType
from pyspark.sql.functions import * 


##We use the ids in string format!
##TO HAVE THE ARRAY OF STRINGS
df_book = df_book\
    .select("id","date", "url", "isbn", "citationscore", "publisher" , "title", split(col("editor_array"),",").alias("editor_array"))
df_incollection = df_incollection\
    .select("id","title","date", "url", "pages", "citationscore", \
    "book_id", split(col("author_array"),",").alias("author_array"), \
    split(col("cited_inc_array"),",").alias("cited_inc_array"))

#df_incollection.printSchema()
#df_incollection.limit(5).select("id", 'author_array').show()
df_incollection = df_incollection \
  .withColumn("id" ,
              df_incollection["id"]
              .cast(StringType()))   \
  .withColumn("title"  ,
              df_incollection["title"]
              .cast(StringType())) \
  .withColumn("date",
              df_incollection["date"]
              .cast(DateType()))    \
  .withColumn("url"  ,
              df_incollection["url"]
              .cast(StringType())) \
  .withColumn("pages"  ,
              df_incollection["pages"]
              .cast(StringType())) \
  .withColumn("citationscore"  ,
              df_incollection["citationscore"]
              .cast(FloatType())) \
#  .withColumn("author_array"  ,
#              df_incollection["author_array"]
#              .cast(ArrayType(StringType()))) \
#  .withColumn("cited_inc_array"  ,
#              df_incollection["cited_inc_array"]
#              .cast(ArrayType(StringType()))) \
#df_incollection.printSchema()
#df_incollection.filter(array_contains(df_incollection.author_array,"7370")).select("id","author_array").show(truncate = False)
#5371|[7371,  7370,  7372]
#df_incollection.select(explode(df_incollection.cited_inc_array)).show()



#df_book.printSchema()
#df_book.show()
df_book = df_book \
  .withColumn("id" ,
              df_book["id"]
              .cast(StringType()))   \
  .withColumn("isbn"  ,
              df_book["isbn"]
              .cast(StringType())) \
  .withColumn("date",
              df_book["date"]
              .cast(DateType()))    \
  .withColumn("url"  ,
              df_book["url"]
              .cast(StringType())) \
  .withColumn("publisher"  ,
              df_book["publisher"]
              .cast(StringType())) \
  .withColumn("citationscore"  ,
              df_book["citationscore"]
              .cast(FloatType())) \
  .withColumn("title"  ,
              df_book["title"]
              .cast(StringType())) \
#  .withColumn("editors_array"  ,
#              df_book_editors["editors_array"]) \
#df_book.printSchema()
#df_book.select(explode(df_book.editor_array)).show()


#df_author.printSchema()
#df_author.limit(5).show()
df_author = df_author \
  .withColumn("id" ,
              df_author["id"]
              .cast(StringType()))   \
  .withColumn("name"  ,
              df_author["name"]
              .cast(StringType())) \
  .withColumn("citationscore",
              df_author["citationscore"]
              .cast(FloatType()))    \
#email is already a string
#df_author.printSchema() 
#df_author.limit(5).show()



In [3]:
#Query complexity : WHERE, GROUP BY
#This query retrieves the top 3 authors for the number of books published after  2000-02-03. To do this a filter condition is applied to 
# the books selecting only the ones published after the chosen date and a group operation is applied to count all books of a certain author.  


df_book.filter(col("date")>lit("2000-02-03").cast(DateType()))\
.groupby(col("publisher"))\
.count()\
.sort(desc("count"))\
.withColumnRenamed("count","number_of_books")\
.show(3,False)

+--------------------+---------------+
|publisher           |number_of_books|
+--------------------+---------------+
|Springer            |57             |
|Chapman and Hall/CRC|18             |
|IGI Global          |9              |
+--------------------+---------------+
only showing top 3 rows



In [4]:

#Query complexity :  WHERE, GROUP BY, HAVING, 2 JOINs 

#This query retrieves the Authors with more then 50 incollections written in all their books written after "2000-02-03"
#To do this the query considers all books written after that date; since a book can be written by multiple authors, the explode method is used to 
#consider each author in the array (editor_array) and perform a join. Another join operation is performed to associate each incollection to its 
# respective book. At the end a group operation is needed to  collect all the incollection related to the books of the same author and count them and apply the filter condition. 

In [5]:
df_book.filter(col("date")>lit("2000-02-03").cast(DateType()))\
.select(df_book.id.alias("BookId"), df_book.date, df_book.title, explode(df_book.editor_array).alias("authorId"))\
.join(df_author, df_author.id == col("authorId"), "inner") \
.drop(col("id"))\
.select(col("BookId"),col("date").alias("PublicationDate"), col("title"), col("authorId"), col("name").alias("AuthorName"))\
.join(df_incollection, df_incollection.book_id == col("BookId"))\
.groupby(col("AuthorId"), col("AuthorName"))\
.count()\
.sort(desc("count"))\
.withColumnRenamed("count","number_of_incollection")\
.where(col("count")>50)\
.select(col("AuthorName"), col("authorId"), col("number_of_incollection")) \
.show(truncate=False)

+-------------------+--------+----------------------+
|AuthorName         |authorId|number_of_incollection|
+-------------------+--------+----------------------+
|Borko Furht        |4881    |76                    |
|Michael ten Hompel |5343    |73                    |
|Thomas Bauernhansl |5342    |73                    |
|Birgit Vogel-Heuser|5341    |73                    |
|Grzegorz Rozenberg |5354    |70                    |
|Lior Rokach        |5220    |58                    |
|Arto Salomaa       |5365    |54                    |
+-------------------+--------+----------------------+



In [25]:
#Query Update
#This query update the email address of an author called "Borko Furht"

df_author.show(20, False)
  
df_author.filter(col("name")=="Borko Furht")\
.show(truncate=False)

ValueWhenTrue = "B.Furht@gmail.com"
df_author=df_author.withColumn('email',when(col("name")=="Borko Furht", ValueWhenTrue).otherwise(col('email')))

df_author.filter(col("name")=="Borko Furht")\
.show(truncate=False)

df_author.show(20, False)

+----+------------------------+-------------+----------------------------------+
|id  |name                    |citationscore|email                             |
+----+------------------------+-------------+----------------------------------+
|2000|Paul Kocher             |0.977108     |paul.kocher@gmail.com             |
|2001|Daniel Genkin           |0.977108     |daniel.genkin@gmail.com           |
|2002|Daniel Gruss            |0.977108     |daniel.gruss@gmail.com            |
|2003|Werner Haas 0004        |0.977108     |werner.haas.0004@gmail.com        |
|2004|Mike Hamburg            |0.977108     |mike.hamburg@gmail.com            |
|2005|Moritz Lipp             |0.977108     |moritz.lipp@gmail.com             |
|2006|Stefan Mangard          |0.977108     |stefan.mangard@gmail.com          |
|2007|Thomas Prescher 0002    |0.977108     |thomas.prescher.0002@gmail.com    |
|2008|Michael Schwarz 0001    |0.977108     |michael.schwarz.0001@gmail.com    |
|2009|Yuval Yarom           